In [ ]:
!pip install datasets
!pip install torch
!pip install transformers
!pip install pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 12.1 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.


In [ ]:
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from transformers import CLIPProcessor, CLIPModel
import torch
import pickle
from PIL import Image
import os
import json
import requests
from io import BytesIO

# Path to the new filtered dataset JSON file
filtered_dataset_path = "/content/drive/MyDrive/products_with_reviews_dataset.json"

# Load the new filtered dataset
with open(filtered_dataset_path, "r") as f:
    filtered_data = json.load(f)

# Load the CLIP model and processor
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
model.eval()

# Helper function to fetch image from URL
def fetch_image_from_url(url):
    try:
        response = requests.get(url)
        response.raise_for_status()
        image = Image.open(BytesIO(response.content)).convert("RGB")
        return image
    except Exception as e:
        print(f"Error fetching image from URL {url}: {e}")
        return None

# Compute embeddings and store metadata
embeddings = []
metadata = []

for i in range(1000):
    record = filtered_data[i]
    image_url = record.get("image", "")
    if image_url:
        try:
            # Fetch the image from the URL
            image = fetch_image_from_url(image_url)
            if image is None:
                continue

            # Prepare the image for the model
            inputs = processor(images=image, return_tensors="pt")
            with torch.no_grad():
                embedding = model.get_image_features(**inputs).squeeze().numpy()

            # Store the embedding and metadata
            embeddings.append(embedding)
            metadata.append({
                "asin": record.get("asin", ""),
                "title": record.get("title", ""),
                "brand": record.get("brand", ""),
                "description": record.get("description", ""),
                "image": image_url,
                "reviews": record.get("reviews", []),
            })
        except Exception as e:
            print(f"Error processing image: {e}")

# Save embeddings and metadata as a pickle file
output_pickle_path = "/content/drive/My Drive/products_with_reviews_image_embeddings.pkl"

# Check if the pickle file exists
if os.path.exists(output_pickle_path):
    # Load existing data
    with open(output_pickle_path, "rb") as f:
        existing_data = pickle.load(f)
    existing_embeddings = existing_data.get("embeddings", [])
    existing_metadata = existing_data.get("metadata", [])
else:
    # Initialize empty data if the file doesn't exist
    existing_embeddings = []
    existing_metadata = []

# Append new embeddings and metadata
combined_embeddings = existing_embeddings + embeddings
combined_metadata = existing_metadata + metadata

# Save the updated data back to the pickle file
with open(output_pickle_path, "wb") as f:
    pickle.dump({"embeddings": combined_embeddings, "metadata": combined_metadata}, f)

print(f"Embeddings and metadata have been saved to {output_pickle_path}")

In [ ]:
import pickle

output_pickle_path = "/content/drive/MyDrive/products_with_reviews_image_embeddings.pkl"

with open(output_pickle_path, "rb") as f:
    data = pickle.load(f)

print(f"Number of embeddings: {len(data['embeddings'])}")
print(f"Embedding{data['embeddings'][0]}")
print(f"Metadata: {data['metadata'][0]}")

Number of embeddings: 999
Embedding[ 1.66256011e-01  2.50418127e-01  4.18588854e-02 -1.94519162e-01
  4.41215225e-02 -1.83077753e-01  1.11410990e-01  3.50771487e-01
 -2.96310037e-01  5.25738001e-01 -1.38003845e-03  2.39235163e-01
  4.89448428e-01 -2.16620266e-02 -4.36632708e-02  4.32968140e-04
 -1.32419765e-01  6.17963731e-01 -3.96626174e-01 -3.52147102e-01
  3.37488592e-01  4.90301579e-01  2.96655595e-01 -2.33362690e-01
  1.53524056e-01  4.23615932e-01 -8.48657936e-02 -1.31361157e-01
 -1.16250649e-01 -5.30903339e-02  9.34586823e-02  4.81514782e-01
 -1.49243355e-01 -1.16727576e-02 -7.61633575e-01 -2.18351960e-01
 -1.86677501e-01 -1.79467797e-01  1.25750646e-01  1.20282912e+00
 -2.69075543e-01 -9.45633799e-02  2.31737308e-02  3.36539894e-02
  3.11464787e-01 -2.00685501e+00  5.26212752e-01  2.45176814e-02
  4.96001750e-01 -4.81013089e-01  1.02596164e-01  2.12482214e-01
  2.60783821e-01  9.11380351e-03 -2.87950277e-01  1.23894460e-01
 -3.03155869e-01  4.39109728e-02  2.43301466e-01  4.202